In [1]:
from astroquery.mast.missions import MastMissions
from astroquery.mast import Observations
import amigo

Observations.login("e9dc6d7cc23344e899e746d9d7df87e0")
missions = MastMissions(mission="jwst")


# Setting data path
from socket import gethostname

print(f"host name: {gethostname()}")

# LOCAL
if (
    gethostname() == "maxs-mbp-14.shared.sydney.edu.au"
    or gethostname() == "Maxs-MacBook-Pro-14.local"
):
    download_dir = "/Volumes/research-data/PRJ-PAT/max/data/JWST/"

# SIH CLUSTER
elif gethostname().startswith("max-"):
    download_dir = "/home/dgxuser/max/data/JWST/"

# OZSTAR
else:
    download_dir = "/fred/oz440/max/data/JWST/"

INFO: MAST API token accepted, welcome Max Charles [astroquery.mast.auth]
host name: Maxs-MacBook-Pro-14.local


In [2]:
DOWNLOAD_FLAG = True
PROCESS_FLAG = True

In [3]:
cols = [
    # "ArchiveFileID",
    # "fileSetName",
    # "productLevel",
    "targprop",
    # "targ_ra",
    # "targ_dec",
    # "instrume",
    # "exp_type",
    # "opticalElements",
    "date_obs",
    "effexptm",
    "program",
    # "observtn",
    # "visit",
    # "publicReleaseDate",
    "pi_name",
    "proposal_type",
    "proposal_cycle",
    "targtype",
    "is_psf",
    # "access",
    # "cal_ver",
    # "ang_sep",
    # "s_region",
]


all_data = missions.query_criteria(
    select_cols=cols,
    exp_type="NIS_AMI",
    program=None,
)

# all_data

In [4]:
from collections import Counter
from datetime import date


def fmt_targets(rows):
    counts = Counter(str(t) for t in rows["targprop"])
    return ", ".join(f"{t} ({n})" for t, n in sorted(counts.items())) or "-"


def fmt_duration(seconds):
    m, s = divmod(round(float(seconds)), 60)
    h, m = divmod(m, 60)
    return f"{h}h {m:02d}m {s:02d}s"


for prog_data in all_data.group_by("program").groups:
    prog = prog_data["program"][0]
    pi = str(prog_data["pi_name"][0])
    cycle = int(prog_data["proposal_cycle"][0])
    ptype = str(prog_data["proposal_type"][0])
    dates = sorted({date.fromisoformat(str(d)[:10]) for d in prog_data["date_obs"]})
    is_cal = prog_data["is_psf"] == "t"

    print(f"\n{prog}  |  {ptype}  |  Cycle {cycle}  |  {pi}")
    print(f"  Dates:       {', '.join(d.strftime('%d-%m-%Y') for d in dates)}")
    print(f"  Exp. time:   {fmt_duration(prog_data['effexptm'].sum())}")
    print(f"  Science:     {fmt_targets(prog_data[~is_cal])}")
    print(f"  Calibrators: {fmt_targets(prog_data[is_cal])}")

    if DOWNLOAD_FLAG:
        # directories
        uncal_dir = f"{download_dir}/{prog}/uncal"
        calslope_dir = f"{download_dir}/{prog}/calslope"

        # downloading
        data_products = missions.get_product_list(prog_data)
        uncal = missions.filter_products(
            data_products,
            extension="uncal.fits",
        )
        manifest = missions.download_products(
            uncal,
            download_dir=f"{download_dir}/{prog}/uncal",
            flat=True,
        )

        # amigo pipeline
        if PROCESS_FLAG:
            amigo.pipelines.process_calslope(uncal_dir, calslope_dir)


1021  |  COM  |  Cycle 0  |  Stansberry, John A.
  Dates:       11-06-2022
  Exp. time:   0h 10m 44s
  Science:     RUMPELSTILZ (2)
  Calibrators: -


ChunkedEncodingError: ("Connection broken: BrokenPipeError(32, 'Broken pipe')", BrokenPipeError(32, 'Broken pipe'))